# Lecture 6: Backpropagation

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 6, with material from Stanford CS231n 2026, Lecture 4.*

To train a network with gradient descent, we need the gradient of the loss with respect to every weight. **Backpropagation** is the algorithm that computes all of them efficiently.

**What you'll learn**
1. Why deriving gradients on paper doesn't scale.
2. **Computational graphs**, and the one rule of backprop: **downstream = upstream × local**.
3. The lecture's examples, worked number by number.
4. Patterns: how add, multiply, copy, and max nodes pass gradients.
5. How to implement backprop: "flat" code, layer objects, and a tiny **autograd** engine.
6. Backprop with **vectors** and **matrices**.

In [ ]:
import math
import numpy as np

## 1. The problem

In Lecture 5 we wrote the gradients of a 2-layer network by hand. That worked, but:
* it's **tedious** (lots of matrix calculus),
* it's **not modular**: change the loss from softmax to SVM and you re-derive everything,
* it's **impossible** for really big models.

**The better idea:** write the computation as a **graph** of simple operations. Each operation only needs to know its own small derivative. Then one backward sweep through the graph, using the chain rule, gives every gradient.

## 2. A simple example

$$f(x, y, z) = (x + y)\cdot z \qquad\text{with } x = -2,\ y = 5,\ z = -4$$

The graph has two nodes: an **add** node computing $q = x + y$, and a **multiply** node computing $f = q\cdot z$.

**Step 1: forward pass.** Compute the values from left to right.

In [ ]:
x, y, z = -2, 5, -4
q = x + y          # add node
f = q * z          # multiply node
print("q =", q, "   f =", f)

**Step 2: backward pass.** Compute the gradients from right to left. Each node's own ("local") derivatives are simple:
* multiply node $f = q\cdot z$: $\ \partial f/\partial q = z$, $\ \partial f/\partial z = q$
* add node $q = x + y$: $\ \partial q/\partial x = 1$, $\ \partial q/\partial y = 1$

Start with $\partial f/\partial f = 1$ and work backward. For $x$, use the **chain rule**: $x$ affects $f$ only through $q$, so

$$\frac{\partial f}{\partial x} = \underbrace{\frac{\partial f}{\partial q}}_{\text{upstream}}\cdot\underbrace{\frac{\partial q}{\partial x}}_{\text{local}}$$

In [ ]:
df_df = 1
df_dz = q * df_df          # local gradient of the multiply node w.r.t. z is q   -> 3
df_dq = z * df_df          # local gradient w.r.t. q is z                       -> -4
df_dx = 1 * df_dq          # add node: local gradient 1, times upstream -4      -> -4
df_dy = 1 * df_dq          #                                                    -> -4
print("df/dx =", df_dx, "  df/dy =", df_dy, "  df/dz =", df_dz)

Check one of them numerically: nudge $x$ a tiny bit and see how $f$ changes.

In [ ]:
h = 1e-6
f_nudged = ((x + h) + y) * z
print("numerical df/dx:", round((f_nudged - f) / h, 4))

## 3. The one rule of backprop

Every node, in any graph, does the same thing:
* **Forward:** take inputs, compute an output.
* **Backward:** receive the **upstream gradient** (how much the final loss changes when this node's output changes), multiply it by the node's **local gradient**, and pass the result **downstream** to its inputs.

$$\text{downstream gradient} = \text{upstream gradient} \times \text{local gradient}$$

Each node only needs its own local information. That's what makes backprop **modular**.

## 4. A bigger example: a sigmoid neuron

$$f(w, x) = \frac{1}{1 + e^{-(w_0x_0 + w_1x_1 + w_2)}} \qquad w_0 = 2,\ x_0 = -1,\ w_1 = -3,\ x_1 = -2,\ w_2 = -3$$

We break it into tiny steps. Local derivatives we'll need:
* $\frac{d}{dx}e^x = e^x$
* $\frac{d}{dx}\frac1x = -\frac{1}{x^2}$
* $\frac{d}{dx}(a\cdot x) = a$
* $\frac{d}{dx}(c + x) = 1$

**Forward pass**, one operation per line:

In [ ]:
w0, x0, w1, x1, w2 = 2, -1, -3, -2, -3

a = w0 * x0            # -2
b = w1 * x1            #  6
c = a + b              #  4
d = c + w2             #  1
e = -d                 # -1
g = math.exp(e)        #  0.37
k = 1 + g              #  1.37
out = 1 / k            #  0.73
print(f"a={a}, b={b}, c={c}, d={d}, e={e}, g={g:.2f}, k={k:.2f}, out={out:.2f}")

**Backward pass**: start at the end with gradient 1. At every step, multiply the upstream gradient by that operation's local derivative.

In [ ]:
d_out = 1.0
d_k = d_out * (-1 / k**2)        # 1/x node: local derivative -1/x^2       -> -0.53
d_g = d_k * 1                    # +1 node: local derivative 1             -> -0.53
d_e = d_g * math.exp(e)          # exp node: local derivative e^x          -> -0.20
d_d = d_e * -1                   # negate node: local derivative -1        ->  0.20
d_c = d_d * 1                    # add node: passes the gradient through   ->  0.20
d_w2 = d_d * 1                   #                                         ->  0.20
d_a = d_c * 1                    # add node again                          ->  0.20
d_b = d_c * 1
d_w0 = d_a * x0                  # multiply node: the "other" input        -> -0.20
d_x0 = d_a * w0                  #                                         ->  0.39
d_w1 = d_b * x1                  #                                         -> -0.39
d_x1 = d_b * w1                  #                                         -> -0.59
print(f"gradients: w0 {d_w0:.2f}, x0 {d_x0:.2f}, w1 {d_w1:.2f}, x1 {d_x1:.2f}, w2 {d_w2:.2f}")

These are the numbers on the lecture slide.

**A shortcut:** we can choose bigger nodes if their local derivative is easy. The last four steps together compute the **sigmoid** $\sigma(x) = 1/(1 + e^{-x})$, and its derivative simplifies to

$$\frac{d\sigma}{dx} = (1 - \sigma(x))\,\sigma(x)$$

so one "sigmoid node" can replace four. Same answer:

In [ ]:
sigma = 1 / (1 + math.exp(-d))
print(f"sigmoid node: local gradient (1 - {sigma:.2f}) * {sigma:.2f} = {(1 - sigma) * sigma:.2f}")

## 5. Patterns in gradient flow

Each kind of node passes gradients in a simple, recognizable way (the lecture's examples):

| Node | What it does to the gradient | Example |
|---|---|---|
| **add** | **distributes**: every input gets the upstream gradient | 3 + 4, upstream 2 → **2 and 2** |
| **multiply** | **swaps**: each input gets upstream × the *other* input | 2 × 3, upstream 5 → **15 and 10** |
| **copy** (a value used twice) | **adds** the upstream gradients | used twice, upstreams 4 and 2 → **6** |
| **max** | **routes**: all of it goes to the larger input | max(4, 5), upstream 9 → **0 and 9** |

In [ ]:
print("add 3 + 4, upstream 2:      ", (2, 2))
print("multiply 2 * 3, upstream 5: ", (5 * 3, 5 * 2))
print("copy, upstreams 4 and 2:    ", 4 + 2)
print("max(4, 5), upstream 9:      ", (0, 9))

**The copy rule is important:** whenever a value is used in several places, its gradients **add up**. That happens all the time, e.g., a CNN filter is used at every image position (Lecture 7), and an RNN reuses its weights at every time step (Lecture 12).

## 6. Implementing backprop

### 6.1 "Flat" code
Write the forward pass line by line, then the backward pass as the **same lines in reverse**. This is how you'll write backward passes in the assignments:

In [ ]:
def sigmoid_neuron(w0, x0, w1, x1, w2):
    # forward pass
    s0 = w0 * x0
    s1 = w1 * x1
    s2 = s0 + s1
    s3 = s2 + w2
    L = 1 / (1 + math.exp(-s3))        # sigmoid
    # backward pass: one line for each forward line, in reverse order
    grad_L = 1.0
    grad_s3 = grad_L * (1 - L) * L     # sigmoid
    grad_w2 = grad_s3                  # add
    grad_s2 = grad_s3
    grad_s0 = grad_s2                  # add
    grad_s1 = grad_s2
    grad_w1 = grad_s1 * x1             # multiply
    grad_x1 = grad_s1 * w1
    grad_w0 = grad_s0 * x0             # multiply
    grad_x0 = grad_s0 * w0
    return L, [grad_w0, grad_x0, grad_w1, grad_x1, grad_w2]

output, gradients = sigmoid_neuron(2, -1, -3, -2, -3)
print(f"output {output:.2f}, gradients {np.round(gradients, 2)}")

### 6.2 Nodes as objects with `forward` and `backward`
Frameworks like PyTorch make each operation an object. `forward` computes the output and **remembers what backward will need**; `backward` turns an upstream gradient into gradients for the inputs.

In [ ]:
class Multiply:
    def forward(self, x, y):
        self.x = x            # remember the inputs for the backward pass
        self.y = y
        return x * y

    def backward(self, upstream):
        grad_x = upstream * self.y      # the "swap" rule
        grad_y = upstream * self.x
        return grad_x, grad_y

node = Multiply()
print("forward(2, 3) =", node.forward(2, 3))
print("backward(5)   =", node.backward(5))

### 6.3 A tiny autograd engine
Now let's make backprop **automatic**, like PyTorch's autograd but for single numbers. Each `Value` remembers:
* its number (`data`) and its gradient (`grad`),
* which values it was computed from (`parents`),
* a small function that applies its local backward rule.

In [ ]:
class Value:
    def __init__(self, data, parents=()):
        self.data = data
        self.grad = 0.0
        self.parents = parents
        self.backward_rule = lambda: None     # filled in by the operation that created this value

    def __add__(self, other):
        out = Value(self.data + other.data, parents=(self, other))
        def backward_rule():
            self.grad += out.grad             # add: pass the gradient through to both inputs
            other.grad += out.grad
        out.backward_rule = backward_rule
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, parents=(self, other))
        def backward_rule():
            self.grad += other.data * out.grad    # multiply: swap the inputs
            other.grad += self.data * out.grad
        out.backward_rule = backward_rule
        return out

Notice the `+=`: that's the copy rule, so a value used twice collects both gradients.

To run backprop, we visit the nodes **from the output back to the inputs**, making sure each node is processed after everything that depends on it (a "topological order"):

In [ ]:
def backward(output):
    # 1. List every value in the graph, each one AFTER the values it was computed from.
    order, visited = [], set()
    def visit(v):
        if v not in visited:
            visited.add(v)
            for parent in v.parents:
                visit(parent)
            order.append(v)
    visit(output)
    # 2. Start with gradient 1 at the output, then apply each node's rule from the end backward.
    output.grad = 1.0
    for v in reversed(order):
        v.backward_rule()

Test it on the first example, $(x + y)\cdot z$, and on a value used twice:

In [ ]:
x, y, z = Value(-2.0), Value(5.0), Value(-4.0)
f = (x + y) * z
backward(f)
print("(x + y) * z:  df/dx =", x.grad, " df/dy =", y.grad, " df/dz =", z.grad)

a = Value(3.0)
g = a * a + a                 # a is used three times; by hand, dg/da = 2a + 1 = 7
backward(g)
print("a * a + a at a = 3:  dg/da =", a.grad)

About 40 lines, and it computes exact gradients for **any** expression built from these operations. PyTorch's autograd is the same algorithm, for arrays instead of single numbers.

## 7. Backprop with vectors

When inputs and outputs are vectors, the "local gradient" is a **Jacobian matrix**: entry $(i, j)$ says how much output $j$ changes when input $i$ changes. The loss is still a single number, so the upstream gradient is a vector of the same size as the output, and:

$$\frac{\partial L}{\partial x} = \frac{\partial z}{\partial x}\,\frac{\partial L}{\partial z}$$

**Important:** the gradient with respect to a variable always has the **same shape as that variable**.

**Example from the lecture: ReLU on a 4-vector.**

In [ ]:
x4 = np.array([1, -2, 3, -1])
z4 = np.maximum(0, x4)
upstream = np.array([4, -1, 5, 9])          # dL/dz

jacobian = np.diag((x4 > 0).astype(float))  # each z_i depends only on x_i: diagonal, 1 where x_i > 0
print("Jacobian dz/dx:\n", jacobian)
print("dL/dx = Jacobian @ upstream =", jacobian @ upstream)

The Jacobian is mostly zeros. **In practice we never build it**; we just apply the rule directly: "pass the upstream gradient through where $x > 0$, else 0":

In [ ]:
print("dL/dx without the Jacobian:", upstream * (x4 > 0))

This matters a lot for real layers: for a batch of 64 vectors of size 4,096, a full Jacobian would have $(64\cdot4096)^2\approx6.9\times10^{10}$ entries, about **256 GB**.

## 8. Backprop with matrices: $y = xw$

The most important layer: matrix multiplication. With $x$ of shape $N\times D$ and $w$ of shape $D\times M$, the output $y = xw$ is $N\times M$. Given the upstream gradient $\partial L/\partial y$ (also $N\times M$):

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}\,w^\top \qquad\qquad \frac{\partial L}{\partial w} = x^\top\,\frac{\partial L}{\partial y}$$

**An easy way to remember them: they're the only ways to make the shapes match.** $\partial L/\partial x$ must be $N\times D$: the only way to get that from an $N\times M$ matrix and $w$ ($D\times M$) is $(N\times M)\cdot(M\times D)$.

The lecture's example:

In [ ]:
x = np.array([[2, 1, -3],
              [-3, 4, 2]])                  # N x D = 2 x 3
w = np.array([[3, 2, 1, -1],
              [2, 1, 3, 2],
              [3, 2, 1, -2]])               # D x M = 3 x 4
dL_dy = np.array([[2, 3, -3, 9],
                  [-8, 1, 4, 6]])           # N x M = 2 x 4

dL_dx = dL_dy @ w.T
dL_dw = x.T @ dL_dy
print("dL/dx =\n", dL_dx, "  shape", dL_dx.shape, "(same as x)")
print("dL/dw shape:", dL_dw.shape, "(same as w)")

Check one entry by hand: $\frac{\partial L}{\partial x_{0,0}} = \sum_m \frac{\partial L}{\partial y_{0,m}}\,w_{0,m} = 2\cdot3 + 3\cdot2 + (-3)\cdot1 + 9\cdot(-1) = 0$. ✔

(One thing to watch out for on the slide: the $y$ values printed there don't actually equal $xw$. For example, $y_{0,0} = 2\cdot3 + 1\cdot2 + (-3)\cdot3 = -1$, not 13. It doesn't matter for the gradients, which only need $x$, $w$, and $\partial L/\partial y$.)

## 9. Putting it together: layers with forward and backward

Now we can build a network from **layers**, each with its own `forward` and `backward`. Training code just runs forward through the layers, then backward in reverse order. Adding a new kind of layer never requires re-deriving the others.

In [ ]:
class Linear:
    def __init__(self, n_in, n_out, rng):
        self.W = rng.normal(0, np.sqrt(2 / n_in), size=(n_in, n_out))
        self.b = np.zeros(n_out)

    def forward(self, x):
        self.x = x                                  # remember the input for backward
        return x @ self.W + self.b

    def backward(self, upstream):
        self.dW = self.x.T @ upstream               # dL/dW = x^T dL/dy
        self.db = upstream.sum(axis=0)
        return upstream @ self.W.T                  # dL/dx = dL/dy W^T


class ReLU:
    def forward(self, x):
        self.positive = x > 0                       # remember where the input was positive
        return x * self.positive

    def backward(self, upstream):
        return upstream * self.positive             # pass the gradient only where x > 0

In [ ]:
def softmax_loss(scores, y):
    # Returns the loss and dL/dscores.
    shifted = scores - scores.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    loss = -np.mean(np.log(probs[np.arange(len(y)), y]))
    d_scores = probs.copy()
    d_scores[np.arange(len(y)), y] -= 1
    return loss, d_scores / len(y)

Build a small network (4 inputs → 5 hidden → 3 classes) and check the gradient of one weight numerically:

In [ ]:
rng = np.random.default_rng(0)
layers = [Linear(4, 5, rng), ReLU(), Linear(5, 3, rng)]
X_small = rng.normal(size=(6, 4))
y_small = np.array([0, 1, 2, 0, 1, 2])

def network_loss():
    out = X_small
    for layer in layers:                 # forward through every layer
        out = layer.forward(out)
    return softmax_loss(out, y_small)

loss, upstream = network_loss()
for layer in reversed(layers):           # backward through every layer, in reverse order
    upstream = layer.backward(upstream)

first_layer = layers[0]
analytic = first_layer.dW[0, 0]
original = first_layer.W[0, 0]
first_layer.W[0, 0] = original + 1e-6; loss_plus, _ = network_loss()
first_layer.W[0, 0] = original - 1e-6; loss_minus, _ = network_loss()
first_layer.W[0, 0] = original
print(f"dL/dW[0,0]: backprop {analytic:.6f}, numerical {(loss_plus - loss_minus) / 2e-6:.6f}")

**That's how every deep learning framework works:** a graph of operations, each with forward and backward, and backprop runs them in reverse.

## 10. In PyTorch (optional, run in Colab)
PyTorch builds the graph **as your code runs** and does the backward pass with `.backward()`:

In [ ]:
# [PyTorch] Needs: pip install torch
import torch
x = torch.tensor(-2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
z = torch.tensor(-4.0, requires_grad=True)
f = (x + y) * z
f.backward()
print("df/dx =", x.grad.item(), " df/dy =", y.grad.item(), " df/dz =", z.grad.item())

## Summary
* Represent the computation as a **graph** of simple operations.
* **Forward pass:** compute values. **Backward pass:** compute gradients in reverse.
* **The rule:** downstream gradient = upstream gradient × local gradient.
* **Patterns:** add distributes, multiply swaps, copy adds, max routes.
* For vectors and matrices, local gradients are Jacobians, but we **never build them**; we use simple rules like $\partial L/\partial x = (\partial L/\partial y)\,w^\top$.
* Gradients always have the **same shape** as the variable.

### Check your understanding
1. In $(x + y)\cdot z$ with $z = -4$, why is $\partial f/\partial x = -4$?
2. A weight is used in two places. How do you combine its two gradients?
3. For $y = xw$ with $x$: 10×20 and $w$: 20×5, what are the shapes of $\partial L/\partial x$ and $\partial L/\partial w$?
4. Why don't we build the Jacobian of a ReLU layer?

<details><summary>Answers</summary>

1. The upstream gradient at $q = x + y$ is $z = -4$, and the add node's local gradient is 1.
2. Add them (the copy rule).
3. Same as the variables: 10×20 and 20×5.
4. It's a huge, mostly-zero diagonal matrix; multiplying by it is the same as zeroing the gradient where the input was negative.
</details>

**Next:** Lecture 7, convolutional networks.